# Phase 4: Final Evaluation Report (Group D)
**Authors:** Crispin Oigara & Victor Kiptoo

This notebook compiles the supervised machine learning results from Group C. We will aggregate the summary metrics, visualize comparative performance, and perform McNemar's Test on the top-performing classifiers to determine statistical significance.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# load each team member's results csv and combine them into one dataframe
result_files = [
    '../results/supervised/brian_results.csv',
    '../results/supervised/lewis_results.csv',
    '../results/supervised/doreen_results.csv',
    '../results/supervised/apollos_results.csv'
]

df_list = [pd.read_csv(f) for f in result_files]
results_df = pd.concat(df_list, ignore_index=True)

# sort from best to worst by F1-Score so we can quickly see the top models
results_df.sort_values(by='F1-Score', ascending=False, inplace=True)
display(results_df)

FileNotFoundError: [Errno 2] No such file or directory: '../results/supervised/brian_results.csv'

In [ ]:
# bar chart comparing all models by their macro F1-Score
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x='F1-Score', y='Model', palette='viridis')
plt.title('Supervised Models Comparison (Macro F1-Score)')
plt.xlabel('Macro F1-Score')
plt.ylabel('Algorithm')
plt.xlim(0.8, 1.0)  # zoom in so the small differences between models are visible
plt.tight_layout()

# save the figure so we can drop it into the final report pdf
plt.savefig('../plots/supervised/model_comparison_f1.png')
plt.show()

## Statistical Significance: McNemar's Test
Several models (Random Forest, SVM, and Gradient Boosting) achieved top-tier accuracy (~0.928). To determine if the difference in their misclassifications is statistically significant, we will conduct McNemar's test.

Because the raw prediction arrays were not exported, we will recreate the test-set predictions for the **Random Forest** and **Gradient Boosting** models using the exact `random_state` defined in the training script.

In [ ]:
# reload the preprocessed dataset to recreate predictions from scratch
df_scaled = pd.read_csv('../data/seeds_preprocessed.csv')
X = df_scaled.drop(columns=['label'])
y = df_scaled['label']

# use the same split parameters from the Group C training script so results match
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# retrain both top models with the same hyperparameters used in Group C
rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

gb = GradientBoostingClassifier(random_state=42, n_estimators=100)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# boolean arrays — True where the model got the prediction right
rf_correct = (y_pred_rf == y_test)
gb_correct = (y_pred_gb == y_test)

# build the 2x2 contingency table needed for McNemar's test
# rows = RF outcome, columns = GB outcome
table = [
    [np.sum(rf_correct & gb_correct),  np.sum(rf_correct & ~gb_correct)],
    [np.sum(~rf_correct & gb_correct), np.sum(~rf_correct & ~gb_correct)]
]

print("Contingency Table (RF vs GB):")
print(np.array(table))

In [ ]:
# run the exact McNemar's test and interpret the result
result = mcnemar(table, exact=True)

print(f"McNemar's Test p-value: {result.pvalue:.4f}")

alpha = 0.05
if result.pvalue < alpha:
    print("Conclusion: There is a statistically significant difference in the model predictions.")
else:
    print("Conclusion: There is NO statistically significant difference in the model predictions. Both models generalize equally well on this test set.")